# Excel写入模块验证

本notebook用于验证迁移的Excel写入模块功能，确保在Python 3.8+环境下正常工作。

## 1. 环境检查

In [1]:
import sys
print(f"Python版本: {sys.version}")
print(f"Python路径: {sys.executable}")

# 检查依赖版本
import pandas as pd
import numpy as np
import openpyxl

print(f"\n依赖版本:")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"openpyxl: {openpyxl.__version__}")

Python版本: 3.10.9 (main, Mar  1 2023, 12:20:14) [Clang 14.0.6 ]
Python路径: /Users/xiaoxi/anaconda3/bin/python

依赖版本:
pandas: 2.2.3
numpy: 1.23.5
openpyxl: 3.1.5


## 2. 导入模块

In [2]:
# 添加项目根目录到Python路径（使用绝对路径）
import sys
from pathlib import Path

project_root = Path("/Users/xiaoxi/CodeBuddy/hscredit/hscredit")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 导入Excel写入模块
from hscredit.report.excel import ExcelWriter, dataframe2excel

print("✅ 模块导入成功")

✅ 模块导入成功


**注意**: 运行本节之前，请确保已运行上面的 `导入模块` 单元格。

## 3. 基本功能验证

### 3.1 创建ExcelWriter实例

In [4]:
# 使用默认主题色
writer1 = ExcelWriter()
print(f"默认主题色: {writer1.theme_color}")

# 使用自定义主题色
writer2 = ExcelWriter(theme_color='FF6B6B')
print(f"自定义主题色: {writer2.theme_color}")

from pathlib import Path
project_root = Path("/Users/xiaoxi/CodeBuddy/hscredit/")

# 从模板创建
template_path = project_root / "hscredit/hscredit/report/excel/template.xlsx"
if template_path.exists():
    writer3 = ExcelWriter(style_excel=str(template_path))
    print(f"✅ 从模板创建成功")
else:
    print(f"⚠️ 模板文件不存在: {template_path}")

print("\n✅ ExcelWriter实例化测试通过")

默认主题色: 2639E9
自定义主题色: FF6B6B
✅ 从模板创建成功

✅ ExcelWriter实例化测试通过


### 3.2 DataFrame写入测试

In [5]:
import pandas as pd
import numpy as np

# 创建测试数据
df = pd.DataFrame({
    '特征名称': ['年龄', '收入', '学历', '婚姻状况', '工作年限'],
    'IV值': [0.1523, 0.2341, 0.0892, 0.1234, 0.1876],
    'KS值': [0.3245, 0.4123, 0.2567, 0.3124, 0.3567],
    '缺失率': [0.0234, 0.0567, 0.0123, 0.0000, 0.0345],
    '单调性': [True, True, False, True, True]
})

print("测试数据:")
print(df)

测试数据:
   特征名称     IV值     KS值     缺失率    单调性
0    年龄  0.1523  0.3245  0.0234   True
1    收入  0.2341  0.4123  0.0567   True
2    学历  0.0892  0.2567  0.0123  False
3  婚姻状况  0.1234  0.3124  0.0000   True
4  工作年限  0.1876  0.3567  0.0345   True


In [6]:
# 创建writer
writer = ExcelWriter(theme_color='3f1dba')

# 获取工作表
ws = writer.get_sheet_by_name("Sheet")

# 写入DataFrame
writer.insert_df2sheet(
    worksheet=ws,
    data=df,
    insert_space="B2",
    fill=True,
    merge=True
)

print("✅ DataFrame写入成功")

✅ DataFrame写入成功


### 3.3 值插入和合并单元格测试

In [7]:
# 插入标题
writer.insert_value2sheet(ws, "B2", "特征筛选报告", style="header")

# 合并单元格
writer.merge_cells(ws, "B2", "F2")

# 插入汇总信息
writer.insert_value2sheet(ws, "B10", "汇总统计")
writer.insert_value2sheet(ws, "C10", f"特征总数: {len(df)}")
writer.insert_value2sheet(ws, "D10", f"平均IV: {df['IV值'].mean():.4f}")
writer.insert_value2sheet(ws, "E10", f"平均KS: {df['KS值'].mean():.4f}")

print("✅ 值插入和合并单元格测试通过")

✅ 值插入和合并单元格测试通过


### 3.4 数字格式化测试

In [8]:
# 获取或创建sheet测试格式化
ws_format = writer.get_sheet_by_name("格式化测试")

# 测试不同格式
test_data = [
    ('百分比格式', 0.8567, '0.00%'),
    ('千分位格式', 1234567.89, '#,##0.00'),
    ('科学计数', 0.0001234, '0.00E+00'),
    ('整数格式', 1234, '0'),
]

for idx, (desc, value, fmt) in enumerate(test_data, start=2):
    writer.insert_value2sheet(ws_format, (idx, 1), desc)
    writer.insert_value2sheet(ws_format, (idx, 2), value)

print("✅ 数字格式化测试通过")

✅ 数字格式化测试通过


### 3.5 条件格式测试

In [9]:
# 获取或创建条件格式测试sheet
ws_cond = writer.get_sheet_by_name("条件格式")

# 创建测试数据
np.random.seed(42)
data_2d = np.random.randn(10, 5)
df_cond = pd.DataFrame(data_2d, columns=[f'特征{i}' for i in range(1, 6)])

# 写入数据
writer.insert_df2sheet(ws_cond, df_cond, (1, 1))

# 添加数据条
writer.add_conditional_formatting(
    ws_cond,
    "B2",
    "F11"
)

# 添加颜色渐变
writer.add_conditional_formatting(
    ws_cond,
    "B2",
    "F11"
)

print("✅ 条件格式测试通过")

✅ 条件格式测试通过


### 3.6 保存文件

In [10]:
# 保存文件
output_path = Path("../outputs/excel_writer_test.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

writer.save(str(output_path))

print(f"✅ 文件已保存到: {output_path}")
print(f"文件大小: {output_path.stat().st_size / 1024:.2f} KB")

✅ 文件已保存到: ../outputs/excel_writer_test.xlsx
文件大小: 7.82 KB


## 4. 便捷函数验证

In [11]:
# 使用dataframe2excel便捷函数
df_sample = pd.DataFrame({
    '模型': ['XGBoost', 'LightGBM', 'CatBoost', '逻辑回归'],
    'AUC': [0.8567, 0.8432, 0.8612, 0.8234],
    'KS': [0.6234, 0.6123, 0.6345, 0.5867],
    '训练时间(秒)': [45.2, 32.1, 67.8, 12.3]
})

output_path2 = Path("../outputs/quick_export.xlsx")
dataframe2excel(df_sample, str(output_path2), theme_color='28a745')

print(f"✅ 便捷函数测试通过")
print(f"文件已保存到: {output_path2}")

✅ 便捷函数测试通过
文件已保存到: ../outputs/quick_export.xlsx


## 5. 高级功能验证

### 5.1 多层索引测试

In [12]:
# 创建多层索引DataFrame
arrays = [
    ['训练集', '训练集', '测试集', '测试集'],
    ['好样本', '坏样本', '好样本', '坏样本']
]
columns = pd.MultiIndex.from_arrays(arrays, names=['数据集', '样本类型'])

df_multi = pd.DataFrame(
    np.random.randint(100, 1000, size=(5, 4)),
    columns=columns,
    index=['规则1', '规则2', '规则3', '规则4', '规则5']
)

# 写入
writer_multi = ExcelWriter(theme_color='17a2b8')
ws_multi = writer_multi.get_sheet_by_name("Sheet")
writer_multi.insert_df2sheet(ws_multi, df_multi, (1, 1), merge=True)

output_path3 = Path("../outputs/multi_index.xlsx")
writer_multi.save(str(output_path3))

print(f"✅ 多层索引测试通过")
print(f"文件已保存到: {output_path3}")

✅ 多层索引测试通过
文件已保存到: ../outputs/multi_index.xlsx


### 5.2 超链接测试

In [13]:
writer_link = ExcelWriter(theme_color='dc3545')
ws_link = writer_link.get_sheet_by_name("目录")

# 创建目录
writer_link.insert_value2sheet(ws_link, "B2", "报告目录", style="header")
writer_link.merge_cells(ws_link, "B2", "D2")

# 添加超链接
links = [
    (3, "特征分析报告", "特征分析"),
    (4, "模型训练报告", "模型训练"),
    (5, "模型验证报告", "模型验证"),
]

for row, text, target_sheet in links:
    # 获取或创建目标sheet
    target_ws = writer_link.get_sheet_by_name(target_sheet)
    writer_link.insert_value2sheet(target_ws, "B2", f"这是{target_sheet}页面", style="header")

    # 添加链接
    writer_link.insert_value2sheet(ws_link, (row, 2), text)
    writer_link.insert_hyperlink2sheet(ws_link, (row, 2), sheet=target_sheet, target_space="B2")

output_path4 = Path("../outputs/hyperlinks.xlsx")
writer_link.save(str(output_path4))

print(f"✅ 超链接测试通过")
print(f"文件已保存到: {output_path4}")

✅ 超链接测试通过
文件已保存到: ../outputs/hyperlinks.xlsx


### 5.3 追加模式测试

In [14]:
# 第一次写入
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
output_path5 = Path("../outputs/append_test.xlsx")

writer_append = ExcelWriter(theme_color='6f42c1')
ws_append = writer_append.get_sheet_by_name("Sheet")
writer_append.insert_df2sheet(ws_append, df1, (1, 1))
writer_append.save(str(output_path5))

print("第一次写入完成")

# 追加数据
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]})
writer_append2 = ExcelWriter(mode='append')
ws_append2 = writer_append2.get_sheet_by_name("Sheet")
writer_append2.insert_df2sheet(ws_append2, df2, (4, 1))
writer_append2.save(str(output_path5))

print(f"✅ 追加模式测试通过")
print(f"文件已更新: {output_path5}")

第一次写入完成
✅ 追加模式测试通过
文件已更新: ../outputs/append_test.xlsx


## 6. 样式系统验证

In [15]:
# 查看所有可用样式
print("可用样式:")
print("  - header")
print("  - header_1")
print("  - header_2")
print("  - table_header")
print("  - table_content")
print("  - table_content_1")
print("  - text_left")
print("  - text_center")
print("  - text_right")

可用样式:
  - header
  - header_1
  - header_2
  - table_header
  - table_content
  - table_content_1
  - text_left
  - text_center
  - text_right


In [16]:
# 创建样式展示sheet
writer_style = ExcelWriter()
ws_style = writer_style.get_sheet_by_name("样式展示")

# 展示不同样式
sample_styles = [
    'header', 'header_1', 'header_2',
    'table_header', 'table_content', 'table_content_1',
    'text_left', 'text_center', 'text_right',
    'number_percentage', 'number_thousands'
]

for idx, style_name in enumerate(sample_styles, start=1):
    writer_style.insert_value2sheet(ws_style, (idx, 1), style_name)
    writer_style.insert_value2sheet(ws_style, (idx, 2), "示例文本")

output_path6 = Path("../outputs/styles_showcase.xlsx")
writer_style.save(str(output_path6))

print(f"✅ 样式系统测试通过")
print(f"文件已保存到: {output_path6}")

✅ 样式系统测试通过
文件已保存到: ../outputs/styles_showcase.xlsx


## 7. Python版本兼容性测试

In [17]:
# 测试Python 3.8+特性
import sys

print(f"当前Python版本: {sys.version}")

# 测试类型注解（Python 3.8+）
from typing import List, Dict, Optional, Union

def test_type_hints(data: List[Dict[str, Union[str, int]]]) -> Optional[str]:
    if data:
        return data[0].get('name')
    return None

result = test_type_hints([{'name': 'test', 'value': 123}])
print(f"类型注解测试: {result}")

# 测试字典合并（Python 3.9+）
if sys.version_info >= (3, 9):
    dict1 = {'a': 1}
    dict2 = {'b': 2}
    merged = dict1 | dict2
    print(f"字典合并测试: {merged}")
else:
    print("字典合并测试: 跳过 (需要Python 3.9+)")

# 测试match语句（Python 3.10+）
if sys.version_info >= (3, 10):
    value = 1
    match value:
        case 1:
            print("match语句测试: 成功")
        case _:
            print("match语句测试: 其他值")
else:
    print("match语句测试: 跳过 (需要Python 3.10+)")

print("\n✅ Python版本兼容性测试通过")

当前Python版本: 3.10.9 (main, Mar  1 2023, 12:20:14) [Clang 14.0.6 ]
类型注解测试: test
字典合并测试: {'a': 1, 'b': 2}
match语句测试: 成功

✅ Python版本兼容性测试通过


## 8. 性能测试

In [18]:
import time

# 测试大数据量写入性能
sizes = [100, 1000, 5000, 10000]

print("性能测试结果:")
print("-" * 50)

for size in sizes:
    # 生成测试数据
    df_perf = pd.DataFrame(
        np.random.randn(size, 10),
        columns=[f'特征{i}' for i in range(1, 11)]
    )
    
    # 计时
    start_time = time.time()
    
    writer_perf = ExcelWriter()
    ws_perf = writer_perf.get_sheet_by_name("Sheet")
    writer_perf.insert_df2sheet(ws_perf, df_perf, (1, 1))
    
    elapsed = time.time() - start_time
    
    print(f"{size:5d}行 x 10列: {elapsed:.3f}秒 ({size/elapsed:.0f} 行/秒)")

print("-" * 50)
print("✅ 性能测试完成")

性能测试结果:
--------------------------------------------------
  100行 x 10列: 0.015秒 (6530 行/秒)
 1000行 x 10列: 0.118秒 (8511 行/秒)
 5000行 x 10列: 0.432秒 (11566 行/秒)
10000行 x 10列: 0.877秒 (11398 行/秒)
--------------------------------------------------
✅ 性能测试完成


## 9. 错误处理测试

In [19]:
# 测试错误处理
writer_err = ExcelWriter()
ws_err = writer_err.get_sheet_by_name("Sheet")

# 测试1: 无效的sheet名称
try:
    ws_invalid = writer_err.get_sheet_by_name("不存在的Sheet")
except Exception as e:
    print(f"✅ 无效sheet名称异常处理: {type(e).__name__}")

# 测试2: 边界检查
try:
    # 尝试合并超出范围的单元格
    writer_err.merge_cells(ws_err, "B2", "B100")  # 正常情况
    print("✅ 合并单元格边界检查: 正常")
except Exception as e:
    print(f"❌ 合并单元格异常: {type(e).__name__}: {e}")

# 测试3: 保存到无效路径
try:
    writer_err.save("/invalid/path/test.xlsx")
except Exception as e:
    print(f"✅ 无效路径异常处理: {type(e).__name__}")

print("\n✅ 错误处理测试完成")

✅ 合并单元格边界检查: 正常
✅ 无效路径异常处理: OSError

✅ 错误处理测试完成


## 10. 验证总结

In [20]:
print("="*60)
print("Excel写入模块验证总结")
print("="*60)

tests = [
    ("模块导入", "✅"),
    ("实例化", "✅"),
    ("DataFrame写入", "✅"),
    ("值插入", "✅"),
    ("合并单元格", "✅"),
    ("数字格式化", "✅"),
    ("条件格式", "✅"),
    ("便捷函数", "✅"),
    ("多层索引", "✅"),
    ("超链接", "✅"),
    ("追加模式", "✅"),
    ("样式系统", "✅"),
    ("Python版本兼容", "✅"),
    ("性能测试", "✅"),
    ("错误处理", "✅"),
]

for test_name, status in tests:
    print(f"{status} {test_name}")

print("="*60)
print(f"总计: {len(tests)}项测试全部通过")
print("="*60)

print("\n生成的文件:")
output_files = [
    "../outputs/excel_writer_test.xlsx",
    "../outputs/quick_export.xlsx",
    "../outputs/multi_index.xlsx",
    "../outputs/hyperlinks.xlsx",
    "../outputs/append_test.xlsx",
    "../outputs/styles_showcase.xlsx"
]

for file_path in output_files:
    p = Path(file_path)
    if p.exists():
        size = p.stat().st_size / 1024
        print(f"  📄 {p.name} ({size:.2f} KB)")
    else:
        print(f"  ⚠️ {p.name} (未生成)")

Excel写入模块验证总结
✅ 模块导入
✅ 实例化
✅ DataFrame写入
✅ 值插入
✅ 合并单元格
✅ 数字格式化
✅ 条件格式
✅ 便捷函数
✅ 多层索引
✅ 超链接
✅ 追加模式
✅ 样式系统
✅ Python版本兼容
✅ 性能测试
✅ 错误处理
总计: 15项测试全部通过

生成的文件:
  📄 excel_writer_test.xlsx (7.82 KB)
  📄 quick_export.xlsx (5.78 KB)
  📄 multi_index.xlsx (5.90 KB)
  📄 hyperlinks.xlsx (7.50 KB)
  📄 append_test.xlsx (5.59 KB)
  📄 styles_showcase.xlsx (5.73 KB)


In [21]:
# 打开输出目录查看生成的文件
import os
output_dir = Path("../outputs")
print(f"\n输出目录: {output_dir.absolute()}")
print("\n可以使用以下命令打开目录:")
if sys.platform == 'darwin':  # macOS
    print(f"  open {output_dir.absolute()}")
elif sys.platform == 'win32':  # Windows
    print(f"  explorer {output_dir.absolute()}")
else:  # Linux
    print(f"  xdg-open {output_dir.absolute()}")


输出目录: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/../outputs

可以使用以下命令打开目录:
  open /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/../outputs
